# ⭐ Day 59: K-Nearest Neighbors (KNN)
## Intuitive Instance-Based Learning | Complete Tutorial with Examples & Exercises

**🎯 Python & AI Learning Path — Day 59 of 369**


## Welcome to Day 59!

Welcome back, learner! Today we dive into one of the most intuitive and elegant algorithms in machine learning: **K-Nearest Neighbors (KNN)**. Unlike many algorithms that build complex models from training data, KNN takes a beautifully simple approach — it makes predictions based on the similarity between new data points and existing examples.

Imagine you're trying to decide if a new fruit is an apple or an orange. You look at its color, size, and texture, then compare it to fruits you already know. If the 5 closest fruits are all apples, you'd probably classify it as an apple too! This is the essence of KNN.

KNN is a **lazy learning** algorithm, meaning it doesn't actively learn a model during training. Instead, it memorizes the training data and performs computations only when making predictions. This makes it incredibly simple to understand and implement, yet surprisingly powerful for many real-world problems.

Today, we'll explore how KNN works under the hood, why **feature scaling** is absolutely critical (spoiler: KNN is distance-based!), how to choose the optimal value of **K**, and when KNN shines versus when other algorithms might be better suited. Let's begin this journey into instance-based learning!


## 📋 Table of Contents

1. [Introduction to K-Nearest Neighbors and Lazy Learning](#1.-introduction-to-k-nearest-neighbors-and-lazy-learning)
2. [How KNN Works (Distance Metrics)](#2.-how-knn-works-distance-metrics)
3. [The Importance of Feature Scaling for KNN](#3.-the-importance-of-feature-scaling-for-knn)
4. [Choosing the Right Value of K](#4.-choosing-the-right-value-of-k)
5. [Implementing KNN with Scikit-Learn](#5.-implementing-knn-with-scikit-learn)
6. [Hyperparameter Tuning](#6.-hyperparameter-tuning)
7. [Visualizing Decision Boundaries](#7.-visualizing-decision-boundaries)
8. [Applying KNN to the Titanic Dataset](#8.-applying-knn-to-the-titanic-dataset)
9. [Pros, Cons, and When to Use KNN](#9.-pros-cons-and-when-to-use-knn)
10. [Comparison with Other Algorithms](#10.-comparison-with-other-algorithms)
11. [🛠️ Hands-On Exercises](#-hands-on-exercises)
12. [Solutions](#solutions)


## 1. Introduction to K-Nearest Neighbors and Lazy Learning

K-Nearest Neighbors is a **non-parametric**, **instance-based** learning algorithm used for both classification and regression tasks.

### Key Characteristics:
- **Lazy Learning**: No explicit training phase; all computation happens at prediction time
- **Instance-Based**: Stores all training examples and uses them directly for predictions
- **Non-Parametric**: Makes no assumptions about the underlying data distribution

### The Algorithm at a Glance:
1. Store all training data
2. For a new data point, calculate distances to all training points
3. Select the K nearest neighbors
4. **Classification**: Majority vote of neighbors' classes
5. **Regression**: Average (or weighted average) of neighbors' values


In [ ]:
# 📦 Import essential libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_moons, load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("✅ Libraries imported successfully!")
print("📊 Ready to explore K-Nearest Neighbors!")


## 2. How KNN Works (Distance Metrics: Euclidean, Manhattan, Minkowski)

KNN relies on **distance metrics** to measure similarity between data points. The choice of distance metric can significantly impact performance.

### Common Distance Metrics:

**Euclidean Distance** (L2 norm): The straight-line distance between two points
$$d(x, y) = \sqrt{\sum_{i=1}^{n}(x_i - y_i)^2}$$

**Manhattan Distance** (L1 norm): The sum of absolute differences along each dimension
$$d(x, y) = \sum_{i=1}^{n}|x_i - y_i|$$

**Minkowski Distance**: Generalization of both Euclidean and Manhattan
$$d(x, y) = \left(\sum_{i=1}^{n}|x_i - y_i|^p\right)^{1/p}$$

- When $p=1$: Manhattan Distance
- When $p=2$: Euclidean Distance


In [ ]:
# 📏 Visualizing different distance metrics
def plot_distance_metrics():
    """Visualize Euclidean vs Manhattan distance in 2D space"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Point A and Point B
    point_a = np.array([1, 1])
    point_b = np.array([4, 5])
    
    # Euclidean Distance
    ax1 = axes[0]
    ax1.scatter(*point_a, color='blue', s=200, label='Point A (1,1)', zorder=5)
    ax1.scatter(*point_b, color='red', s=200, label='Point B (4,5)', zorder=5)
    ax1.plot([point_a[0], point_b[0]], [point_a[1], point_b[1]], 
             'g--', linewidth=3, label='Euclidean Path')
    
    euclidean_dist = np.sqrt(np.sum((point_a - point_b)**2))
    ax1.set_title(f'Euclidean Distance (L2)\nDistance = {euclidean_dist:.2f}', fontsize=14)
    ax1.set_xlim(0, 6)
    ax1.set_ylim(0, 6)
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    ax1.set_aspect('equal')
    
    # Manhattan Distance
    ax2 = axes[1]
    ax2.scatter(*point_a, color='blue', s=200, label='Point A (1,1)', zorder=5)
    ax2.scatter(*point_b, color='red', s=200, label='Point B (4,5)', zorder=5)
    
    # Manhattan path (L-shaped)
    ax2.plot([point_a[0], point_b[0]], [point_a[1], point_a[1]], 
             'orange', linewidth=3, label='Manhattan Path')
    ax2.plot([point_b[0], point_b[0]], [point_a[1], point_b[1]], 
             'orange', linewidth=3)
    
    manhattan_dist = np.sum(np.abs(point_a - point_b))
    ax2.set_title(f'Manhattan Distance (L1)\nDistance = {manhattan_dist:.2f}', fontsize=14)
    ax2.set_xlim(0, 6)
    ax2.set_ylim(0, 6)
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    ax2.set_aspect('equal')
    
    plt.suptitle('Distance Metrics Comparison', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_distance_metrics()


In [ ]:
# 💡 Manual implementation of KNN to understand the algorithm
class ManualKNN:
    """A simple manual implementation of KNN for educational purposes"""
    
    def __init__(self, k=3, metric='euclidean'):
        self.k = k
        self.metric = metric
    
    def fit(self, X, y):
        """Store training data"""
        self.X_train = X
        self.y_train = y
    
    def _calculate_distance(self, x1, x2):
        """Calculate distance between two points"""
        if self.metric == 'euclidean':
            return np.sqrt(np.sum((x1 - x2)**2))
        elif self.metric == 'manhattan':
            return np.sum(np.abs(x1 - x2))
        else:
            raise ValueError("Unsupported metric")
    
    def predict_single(self, x):
        """Predict class for a single data point"""
        # Calculate distances to all training points
        distances = [self._calculate_distance(x, x_train) 
                    for x_train in self.X_train]
        
        # Get indices of k nearest neighbors
        k_indices = np.argsort(distances)[:self.k]
        
        # Get labels of k nearest neighbors
        k_nearest_labels = [self.y_train[i] for i in k_indices]
        
        # Return majority vote
        return max(set(k_nearest_labels), key=k_nearest_labels.count)
    
    def predict(self, X):
        """Predict classes for multiple data points"""
        return np.array([self.predict_single(x) for x in X])

# Test manual implementation
X_simple = np.array([[1, 2], [2, 3], [3, 3], [6, 7], [7, 8], [8, 8]])
y_simple = np.array([0, 0, 0, 1, 1, 1])

knn_manual = ManualKNN(k=3)
knn_manual.fit(X_simple, y_simple)

test_point = np.array([[5, 5]])
prediction = knn_manual.predict(test_point)

print("🔍 Manual KNN Test:")
print(f"Training data shape: {X_simple.shape}")
print(f"Test point: {test_point[0]}")
print(f"Prediction: Class {prediction[0]}")
print(f"Expected: Class 1 (closer to red cluster)")


## 3. The Importance of Feature Scaling for KNN

⚠️ **CRITICAL**: KNN is a distance-based algorithm, which means it's highly sensitive to the scale of features!

If one feature ranges from 0-1000 and another from 0-1, the first feature will dominate the distance calculation, regardless of its actual importance.

### Solutions:
- **Standardization** (Z-score normalization): $x' = \frac{x - \mu}{\sigma}$
- **Min-Max Scaling**: $x' = \frac{x - x_{min}}{x_{max} - x_{min}}$


In [ ]:
# 📊 Demonstrating why scaling matters for KNN
# Create synthetic data with different scales
np.random.seed(42)

# Feature 1: Income (thousands) - Range: 20-150
# Feature 2: Age (years) - Range: 18-70
n_samples = 200
income = np.random.uniform(20, 150, n_samples)
age = np.random.uniform(18, 70, n_samples)

# Create target: 0 = Standard, 1 = Premium (based on income threshold)
target = (income > 85).astype(int)

X_unscaled = np.column_stack([income, age])
y = target

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_unscaled, y, test_size=0.3, random_state=42)

# Train KNN without scaling
knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_unscaled.fit(X_train, y_train)
acc_unscaled = accuracy_score(y_test, knn_unscaled.predict(X_test))

# Train KNN with scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
acc_scaled = accuracy_score(y_test, knn_scaled.predict(X_test_scaled))

print("🔍 Feature Scaling Impact on KNN:")
print(f"Without Scaling: {acc_unscaled:.3f} accuracy")
print(f"With Scaling: {acc_scaled:.3f} accuracy")
print(f"Improvement: {acc_scaled - acc_unscaled:.3f}")


In [ ]:
# 📈 Visualize the scaling effect
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Before scaling
ax1 = axes[0]
scatter1 = ax1.scatter(X_train[:, 0], X_train[:, 1], c=y_train, 
                       cmap='coolwarm', alpha=0.6, edgecolors='black')
ax1.set_xlabel('Income (thousands)', fontsize=12)
ax1.set_ylabel('Age (years)', fontsize=12)
ax1.set_title('Before Scaling\n(X-axis dominates due to larger scale)', fontsize=13)
ax1.set_xlim(0, 160)
ax1.set_ylim(15, 75)
plt.colorbar(scatter1, ax=ax1, label='Class')

# After scaling
ax2 = axes[1]
scatter2 = ax2.scatter(X_train_scaled[:, 0], X_train_scaled[:, 1], c=y_train, 
                       cmap='coolwarm', alpha=0.6, edgecolors='black')
ax2.set_xlabel('Income (standardized)', fontsize=12)
ax2.set_ylabel('Age (standardized)', fontsize=12)
ax2.set_title('After Scaling\n(Both features contribute equally)', fontsize=13)
plt.colorbar(scatter2, ax=ax2, label='Class')

plt.suptitle('Feature Scaling: Critical for KNN Performance', 
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Key Insight:")
print("Without scaling, the Income feature (20-150) dominates Age (18-70)")
print("After scaling, both features contribute fairly to distance calculations!")


## 4. Choosing the Right Value of K (Bias-Variance Tradeoff)

The choice of K is crucial and represents a classic **bias-variance tradeoff**:

- **Small K** (e.g., K=1): 
  - Low bias, high variance (overfitting)
  - Very sensitive to noise
  - Complex decision boundaries

- **Large K** (e.g., K=50):
  - High bias, low variance (underfitting)
  - Smoother decision boundaries
  - May include points from other classes

### How to choose K?
1. **Cross-validation**: Try multiple values and pick the best
2. **Elbow Method**: Plot error vs K and look for the "elbow"
3. **Rule of thumb**: $K = \sqrt{n}$ where n is number of samples
4. **Odd numbers**: Use odd K to avoid ties in binary classification


In [ ]:
# 🔍 Finding optimal K using validation curve
# Generate synthetic dataset
X, y = make_classification(n_samples=500, n_features=2, n_redundant=0, 
                           n_informative=2, n_clusters_per_class=1, 
                           random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Test different K values
k_values = range(1, 31)
train_scores = []
test_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    train_scores.append(knn.score(X_train_scaled, y_train))
    test_scores.append(knn.score(X_test_scaled, y_test))

# Plot validation curve
plt.figure(figsize=(12, 6))
plt.plot(k_values, train_scores, 'b-o', label='Training Accuracy', markersize=6)
plt.plot(k_values, test_scores, 'r-s', label='Test Accuracy', markersize=6)
plt.xlabel('Number of Neighbors (K)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('KNN Validation Curve: Finding Optimal K', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

# Find optimal K
optimal_k = k_values[np.argmax(test_scores)]
plt.axvline(x=optimal_k, color='green', linestyle='--', 
            label=f'Optimal K = {optimal_k}')
plt.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f"✅ Optimal K value: {optimal_k}")
print(f"Best Test Accuracy: {max(test_scores):.4f}")


## 5. Implementing KNN with Scikit-Learn (KNeighborsClassifier & KNeighborsRegressor)

Scikit-Learn provides powerful and efficient implementations of KNN:

- **KNeighborsClassifier**: For classification tasks
- **KNeighborsRegressor**: For regression tasks (predicts average of neighbors)


In [ ]:
# 🎯 KNN Classification Example with Iris Dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42)

# Scale features
scaler_iris = StandardScaler()
X_train_iris_scaled = scaler_iris.fit_transform(X_train_iris)
X_test_iris_scaled = scaler_iris.transform(X_test_iris)

# Train KNN Classifier
knn_clf = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn_clf.fit(X_train_iris_scaled, y_train_iris)

# Predictions
y_pred_iris = knn_clf.predict(X_test_iris_scaled)

print("🌸 KNN Classification Results (Iris Dataset):")
print(f"Accuracy: {accuracy_score(y_test_iris, y_pred_iris):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_iris, y_pred_iris, 
                           target_names=iris.target_names))


In [ ]:
# 📈 KNN Regression Example
from sklearn.datasets import make_regression

# Create regression dataset
X_reg, y_reg = make_regression(n_samples=200, n_features=1, noise=15, random_state=42)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42)

# Train KNN Regressor with different K values
k_values_reg = [1, 5, 15, 30]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, k in enumerate(k_values_reg):
    knn_reg = KNeighborsRegressor(n_neighbors=k)
    knn_reg.fit(X_train_reg, y_train_reg)
    
    # Sort for smooth plotting
    X_sorted = np.sort(X_test_reg, axis=0)
    y_pred_sorted = knn_reg.predict(X_sorted)
    
    ax = axes[idx]
    ax.scatter(X_train_reg, y_train_reg, c='blue', alpha=0.4, label='Training Data')
    ax.scatter(X_test_reg, y_test_reg, c='red', alpha=0.6, label='Test Data')
    ax.plot(X_sorted, y_pred_sorted, 'g-', linewidth=2, label=f'KNN Prediction (K={k})')
    ax.set_title(f'KNN Regression (K={k})', fontsize=12)
    ax.set_xlabel('Feature')
    ax.set_ylabel('Target')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('KNN Regression: Effect of K on Model Flexibility', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Observation:")
print("K=1: Overfitting (follows every data point)")
print("K=30: Underfitting (too smooth, misses patterns)")
print("K=5-15: Good balance")


## 6. Hyperparameter Tuning (n_neighbors, weights, metric)

KNN has several important hyperparameters:

### Key Hyperparameters:
1. **n_neighbors (K)**: Number of neighbors to consider
2. **weights**: How to weight neighbors
   - `'uniform'`: All neighbors weighted equally
   - `'distance'`: Closer neighbors have more weight
3. **metric**: Distance metric to use
   - `'euclidean'`, `'manhattan'`, `'minkowski'`, `'chebyshev'`, etc.
4. **p**: Power parameter for Minkowski metric
   - p=1: Manhattan, p=2: Euclidean


In [ ]:
# 🔧 Grid Search for optimal hyperparameters
param_grid = {
    'n_neighbors': range(1, 21),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

knn_grid = KNeighborsClassifier()
grid_search = GridSearchCV(knn_grid, param_grid, cv=5, scoring='accuracy', n_jobs=-1)

grid_search.fit(X_train_scaled, y_train)

print("🎯 Best Hyperparameters:")
print(f"Best K: {grid_search.best_params_['n_neighbors']}")
print(f"Best Weights: {grid_search.best_params_['weights']}")
print(f"Best Metric: {grid_search.best_params_['metric']}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

# Test on holdout set
best_knn = grid_search.best_estimator_
test_accuracy = best_knn.score(X_test_scaled, y_test)
print(f"\nTest Set Accuracy: {test_accuracy:.4f}")


## 7. Visualizing Decision Boundaries on 2D Data

Decision boundaries help us understand how KNN partitions the feature space. Let's visualize how different K values affect the decision boundary.


In [ ]:
# 🎨 Visualize decision boundaries for different K values
def plot_decision_boundaries(X, y, k_values, title_suffix=""):
    """Plot decision boundaries for different K values"""
    
    # Scale data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Create mesh grid
    h = 0.02
    x_min, x_max = X_scaled[:, 0].min() - 1, X_scaled[:, 0].max() + 1
    y_min, y_max = X_scaled[:, 1].min() - 1, X_scaled[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    n_plots = len(k_values)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    
    for idx, k in enumerate(k_values):
        # Train KNN
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_scaled, y)
        
        # Predict on mesh
        Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
        Z = Z.reshape(xx.shape)
        
        # Plot
        ax = axes[idx]
        ax.contourf(xx, yy, Z, alpha=0.4, cmap='coolwarm')
        scatter = ax.scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, 
                           cmap='coolwarm', edgecolors='black', s=50)
        ax.set_title(f'K = {k}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Feature 1 (scaled)')
        ax.set_ylabel('Feature 2 (scaled)')
    
    plt.suptitle(f'KNN Decision Boundaries {title_suffix}', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Moons dataset (non-linear boundary)
X_moons, y_moons = make_moons(n_samples=300, noise=0.3, random_state=42)
plot_decision_boundaries(X_moons, y_moons, [1, 5, 15, 30], 
                         "(Moons Dataset - Non-linear)")


In [ ]:
# 🎨 Compare uniform vs distance weighting
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

X_demo, y_demo = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                     n_informative=2, n_clusters_per_class=1,
                                     random_state=42)
scaler_demo = StandardScaler()
X_demo_scaled = scaler_demo.fit_transform(X_demo)

# Create mesh
h = 0.02
x_min, x_max = X_demo_scaled[:, 0].min() - 1, X_demo_scaled[:, 0].max() + 1
y_min, y_max = X_demo_scaled[:, 1].min() - 1, X_demo_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

configs = [
    (5, 'uniform', 'K=5, Uniform Weights'),
    (5, 'distance', 'K=5, Distance Weights'),
    (15, 'uniform', 'K=15, Uniform Weights'),
    (15, 'distance', 'K=15, Distance Weights')
]

for idx, (k, weight, title) in enumerate(configs):
    ax = axes[idx // 2, idx % 2]
    
    knn = KNeighborsClassifier(n_neighbors=k, weights=weight)
    knn.fit(X_demo_scaled, y_demo)
    
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.4, cmap='coolwarm')
    ax.scatter(X_demo_scaled[:, 0], X_demo_scaled[:, 1], c=y_demo,
               cmap='coolwarm', edgecolors='black', s=40)
    ax.set_title(title, fontsize=12, fontweight='bold')

plt.suptitle('Effect of Weighting Strategy on Decision Boundaries', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 8. Applying KNN to the Titanic Dataset

Let's apply KNN to a real-world dataset — the famous Titanic survival prediction problem!


In [ ]:
# 🚢 Load and preprocess Titanic dataset
# Note: Using a simplified version for demonstration
from sklearn.datasets import fetch_openml

try:
    # Load Titanic dataset
    titanic = fetch_openml('titanic', version=1, as_frame=True, parser='auto')
    df = titanic.frame
    
    # Select relevant features
    features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']
    df_subset = df[features + ['survived']].copy()
    
    # Encode sex
    df_subset['sex'] = df_subset['sex'].map({'male': 0, 'female': 1})
    
    # Handle missing values
    df_subset['age'] = df_subset['age'].fillna(df_subset['age'].median())
    df_subset['fare'] = df_subset['fare'].fillna(df_subset['fare'].median())
    
    X_titanic = df_subset[features]
    y_titanic = df_subset['survived'].astype(int)
    
    print("✅ Titanic Dataset Loaded Successfully!")
    print(f"Dataset shape: {X_titanic.shape}")
    print(f"\nFeature preview:")
    print(X_titanic.head())
    
except Exception as e:
    # Create synthetic Titanic-like data if fetch fails
    print("Creating synthetic Titanic-like dataset...")
    np.random.seed(42)
    n = 891
    
    X_titanic = pd.DataFrame({
        'pclass': np.random.choice([1, 2, 3], n, p=[0.24, 0.21, 0.55]),
        'sex': np.random.choice([0, 1], n, p=[0.65, 0.35]),
        'age': np.random.normal(29, 14, n).clip(0.4, 80),
        'sibsp': np.random.choice([0, 1, 2, 3, 4, 5, 8], n, p=[0.68, 0.23, 0.03, 0.02, 0.02, 0.01, 0.01]),
        'parch': np.random.choice([0, 1, 2, 3, 4, 5, 6], n, p=[0.76, 0.13, 0.09, 0.01, 0.01, 0.01, 0.01]),
        'fare': np.random.exponential(32, n).clip(0, 512)
    })
    
    # Create target with realistic patterns
    survival_prob = 0.1 + 0.3 * X_titanic['sex'] + 0.2 * (X_titanic['pclass'] == 1) + 0.1 * (X_titanic['age'] < 16)
    y_titanic = (np.random.random(n) < survival_prob).astype(int)
    
    print("✅ Synthetic Titanic Dataset Created!")
    print(f"Dataset shape: {X_titanic.shape}")
    print(f"\nFeature preview:")
    print(X_titanic.head())


In [ ]:
# 🚢 Train and evaluate KNN on Titanic data
X_train_titanic, X_test_titanic, y_train_titanic, y_test_titanic = train_test_split(
    X_titanic, y_titanic, test_size=0.2, random_state=42, stratify=y_titanic)

# Scale features
scaler_titanic = StandardScaler()
X_train_titanic_scaled = scaler_titanic.fit_transform(X_train_titanic)
X_test_titanic_scaled = scaler_titanic.transform(X_test_titanic)

# Train KNN
knn_titanic = KNeighborsClassifier(n_neighbors=7, weights='distance')
knn_titanic.fit(X_train_titanic_scaled, y_train_titanic)

# Predictions
y_pred_titanic = knn_titanic.predict(X_test_titanic_scaled)

print("🚢 KNN Results on Titanic Dataset:")
print(f"Accuracy: {accuracy_score(y_test_titanic, y_pred_titanic):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_titanic, y_pred_titanic, 
                           target_names=['Not Survived', 'Survived']))

# Confusion Matrix
cm = confusion_matrix(y_test_titanic, y_pred_titanic)
print(f"\nConfusion Matrix:")
print(cm)


## 9. Pros, Cons, and When to Use KNN

### ✅ Pros:
- **Simple and intuitive**: Easy to understand and implement
- **No training phase**: Lazy learning means no model training time
- **Non-parametric**: Makes no assumptions about data distribution
- **Handles multi-class naturally**: No need for one-vs-rest strategies
- **Adaptable**: Can be used for both classification and regression

### ❌ Cons:
- **Slow prediction**: Must calculate distance to all training points
- **Curse of dimensionality**: Performance degrades in high dimensions
- **Sensitive to irrelevant features**: All features contribute to distance
- **Requires feature scaling**: Essential for meaningful distances
- **Memory intensive**: Stores entire training dataset

### 🎯 When to Use KNN:
- Small to medium-sized datasets
 - Low-dimensional feature spaces
- When interpretability is important
- When the decision boundary is highly irregular
- Recommendation systems (collaborative filtering)


## 10. Comparison with Other Algorithms (Logistic Regression, SVM, Decision Tree)

Let's compare KNN with other popular classification algorithms!


In [ ]:
# 📊 Compare KNN with other algorithms
algorithms = {
    'KNN (K=5)': KNeighborsClassifier(n_neighbors=5),
    'KNN (K=7, Distance)': KNeighborsClassifier(n_neighbors=7, weights='distance'),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42)
}

results = []

for name, model in algorithms.items():
    # Cross-validation scores
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    
    # Fit and test
    model.fit(X_train_scaled, y_train)
    test_score = model.score(X_test_scaled, y_test)
    
    results.append({
        'Algorithm': name,
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Test Accuracy': test_score
    })

results_df = pd.DataFrame(results)
print("📊 Algorithm Comparison:")
print(results_df.to_string(index=False))

# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(len(results_df))

bars = ax.bar(x_pos, results_df['Test Accuracy'], 
              yerr=results_df['CV Std'], capsize=5,
              color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7'],
              edgecolor='black', linewidth=1.5)

ax.set_xlabel('Algorithm', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Algorithm Comparison on Test Set', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(results_df['Algorithm'], rotation=15, ha='right')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, val in zip(bars, results_df['Test Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


## 🛠️ Hands-On Exercises

Now it's your turn to practice! Complete these exercises to solidify your understanding of KNN.

### Exercise 1: Implement KNN Manually
Implement a KNN classifier from scratch using only NumPy. Test it on a simple 2D dataset.


In [ ]:
# Exercise 1: Your code here



### Exercise 2: Effect of Different K Values
Train KNN classifiers with K = 1, 3, 5, 10, 20, 50 on the Iris dataset. Plot the accuracy vs K and identify the optimal value.


In [ ]:
# Exercise 2: Your code here



### Exercise 3: Visualize Decision Boundaries
Create a visualization showing decision boundaries for K = 1, 5, 15, 30 on the make_moons dataset. Observe how the boundary changes.


In [ ]:
# Exercise 3: Your code here



### Exercise 4: Demonstrate Feature Scaling Importance
Create a dataset where features have vastly different scales. Compare KNN performance before and after scaling. Show that accuracy improves significantly with scaling.


In [ ]:
# Exercise 4: Your code here



### Exercise 5: Distance Metrics Comparison
Compare Euclidean, Manhattan, and Minkowski (p=3) distance metrics on the same dataset. Which performs best?


In [ ]:
# Exercise 5: Your code here



### Exercise 6: Weighted vs Uniform KNN
Compare uniform weighting vs distance-based weighting. Visualize how decision boundaries differ between the two approaches.


In [ ]:
# Exercise 6: Your code here



### Exercise 7: Hyperparameter Tuning with GridSearchCV
Use GridSearchCV to find the optimal combination of n_neighbors, weights, and metric for the Titanic dataset.


In [ ]:
# Exercise 7: Your code here



### Exercise 8: KNN for Regression
Use KNeighborsRegressor to predict a continuous target variable. Experiment with different K values and observe the smoothing effect.


In [ ]:
# Exercise 8: Your code here



### Exercise 9: KNN vs Other Algorithms
Compare KNN with Logistic Regression and SVM on the same dataset. Use cross-validation and create a comparison table of mean accuracy and standard deviation.


In [ ]:
# Exercise 9: Your code here



### Exercise 10: Curse of Dimensionality
Demonstrate how KNN performance degrades as the number of features increases. Create datasets with 2, 10, 50, and 100 features and compare KNN accuracy.


In [ ]:
# Exercise 10: Your code here



## Solutions (Review After Attempting)

Below are detailed solutions for each exercise. Review these only after attempting the exercises yourself!


In [ ]:
# 💡 Solution 1: Manual KNN Implementation
class MyKNN:
    def __init__(self, k=3):
        self.k = k
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
    
    def predict(self, X):
        predictions = []
        for x in X:
            # Calculate distances
            distances = np.sqrt(np.sum((self.X_train - x)**2, axis=1))
            # Get k nearest
            k_indices = np.argsort(distances)[:self.k]
            k_labels = self.y_train[k_indices]
            # Majority vote
            pred = max(set(k_labels), key=list(k_labels).count)
            predictions.append(pred)
        return np.array(predictions)

# Test
X_test = np.array([[2, 3], [5, 6]])
y_test = np.array([0, 1])
my_knn = MyKNN(k=3)
my_knn.fit(X_simple, y_simple)
print("✅ Manual KNN working! Predictions:", my_knn.predict(X_test))


In [ ]:
# 💡 Solution 2: Effect of K Values
k_range = [1, 3, 5, 10, 20, 50]
accuracies = []

X_iris_train, X_iris_test, y_iris_train, y_iris_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_iris_train_s = scaler.fit_transform(X_iris_train)
X_iris_test_s = scaler.transform(X_iris_test)

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_iris_train_s, y_iris_train)
    acc = knn.score(X_iris_test_s, y_iris_test)
    accuracies.append(acc)
    print(f"K={k}: Accuracy={acc:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(k_range, accuracies, 'bo-')
plt.xlabel('K')
plt.ylabel('Accuracy')
plt.title('KNN Accuracy vs K Value (Iris Dataset)')
plt.grid(True)
plt.show()


In [ ]:
# 💡 Solution 3: Decision Boundaries Visualization
def plot_knn_boundaries(X, y, k_values):
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)
    
    h = 0.02
    x_min, x_max = X_s[:, 0].min() - 1, X_s[:, 0].max() + 1
    y_min, y_max = X_s[:, 1].min() - 1, X_s[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    fig, axes = plt.subplots(1, len(k_values), figsize=(16, 4))
    
    for idx, k in enumerate(k_values):
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_s, y)
        Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        
        axes[idx].contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
        axes[idx].scatter(X_s[:, 0], X_s[:, 1], c=y, cmap='RdYlBu', edgecolors='k')
        axes[idx].set_title(f'K = {k}')
    
    plt.tight_layout()
    plt.show()

X_moons, y_moons = make_moons(n_samples=300, noise=0.3, random_state=42)
plot_knn_boundaries(X_moons, y_moons, [1, 5, 15, 30])


In [ ]:
# 💡 Solution 4: Feature Scaling Importance
np.random.seed(42)
n = 200
feature_large = np.random.uniform(0, 1000, n)
feature_small = np.random.uniform(0, 1, n)
y_scale = (feature_large > 500).astype(int)

X_scale = np.column_stack([feature_large, feature_small])
X_tr, X_te, y_tr, y_te = train_test_split(X_scale, y_scale, test_size=0.3, random_state=42)

# Without scaling
knn_no_scale = KNeighborsClassifier(n_neighbors=5)
knn_no_scale.fit(X_tr, y_tr)
acc_no_scale = knn_no_scale.score(X_te, y_te)

# With scaling
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)
knn_scale = KNeighborsClassifier(n_neighbors=5)
knn_scale.fit(X_tr_s, y_tr)
acc_scale = knn_scale.score(X_te_s, y_te)

print(f"Without Scaling: {acc_no_scale:.4f}")
print(f"With Scaling: {acc_scale:.4f}")
print(f"Improvement: {acc_scale - acc_no_scale:.4f}")


In [ ]:
# 💡 Solution 5: Distance Metrics Comparison
metrics = ['euclidean', 'manhattan', 'minkowski']
results = []

for metric in metrics:
    if metric == 'minkowski':
        knn = KNeighborsClassifier(n_neighbors=5, metric=metric, p=3)
    else:
        knn = KNeighborsClassifier(n_neighbors=5, metric=metric)
    
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
    results.append({'Metric': metric, 'Mean CV Score': scores.mean()})

results_df = pd.DataFrame(results)
print(results_df)


In [ ]:
# 💡 Solution 6: Weighted vs Uniform
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

X_w, y_w = make_classification(n_samples=200, n_features=2, n_redundant=0,
                               n_informative=2, random_state=42)
X_w_s = StandardScaler().fit_transform(X_w)

for idx, weight in enumerate(['uniform', 'distance']):
    knn = KNeighborsClassifier(n_neighbors=7, weights=weight)
    knn.fit(X_w_s, y_w)
    
    h = 0.02
    x_min, x_max = X_w_s[:, 0].min() - 1, X_w_s[:, 0].max() + 1
    y_min, y_max = X_w_s[:, 1].min() - 1, X_w_s[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    axes[idx].contourf(xx, yy, Z, alpha=0.4)
    axes[idx].scatter(X_w_s[:, 0], X_w_s[:, 1], c=y_w, edgecolors='k')
    axes[idx].set_title(f'Weights: {weight}')

plt.show()


In [ ]:
# 💡 Solution 7: GridSearchCV
param_grid = {
    'n_neighbors': range(1, 21),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='accuracy')
grid.fit(X_train_titanic_scaled, y_train_titanic)

print("Best parameters:", grid.best_params_)
print("Best CV score:", grid.best_score_)
print("Test score:", grid.score(X_test_titanic_scaled, y_test_titanic))


In [ ]:
# 💡 Solution 8: KNN Regression
from sklearn.datasets import make_regression

X_reg, y_reg = make_regression(n_samples=100, n_features=1, noise=10, random_state=42)
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42)

k_values = [1, 3, 10]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, k in enumerate(k_values):
    knn_reg = KNeighborsRegressor(n_neighbors=k)
    knn_reg.fit(X_reg_train, y_reg_train)
    
    X_plot = np.linspace(X_reg.min(), X_reg.max(), 100).reshape(-1, 1)
    y_pred = knn_reg.predict(X_plot)
    
    axes[idx].scatter(X_reg_train, y_reg_train, alpha=0.5, label='Train')
    axes[idx].plot(X_plot, y_pred, 'r-', linewidth=2, label='Prediction')
    axes[idx].set_title(f'KNN Regression (K={k})')
    axes[idx].legend()

plt.tight_layout()
plt.show()


In [ ]:
# 💡 Solution 9: Algorithm Comparison
models = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVM': SVC(kernel='rbf')
}

comparison = []
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    comparison.append({
        'Model': name,
        'Mean Accuracy': scores.mean(),
        'Std': scores.std()
    })

print(pd.DataFrame(comparison))


In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# 💡 Solution 10: Curse of Dimensionality
dimensions = [2, 10, 50, 100]
accuracies = []

for dim in dimensions:
    # 1. n_redundant=0 (fixed the last error)
    # 2. n_clusters_per_class=1 (fixes this error for dim=2)
    X_dim, y_dim = make_classification(
        n_samples=500, 
        n_features=dim, 
        n_informative=max(2, dim//2), # Ensures at least 2 informative features
        n_redundant=0, 
        n_clusters_per_class=1, 
        random_state=42
    )
    
    X_d_train, X_d_test, y_d_train, y_d_test = train_test_split(
        X_dim, y_dim, test_size=0.3, random_state=42)
    
    scaler = StandardScaler()
    X_d_train_s = scaler.fit_transform(X_d_train)
    X_d_test_s = scaler.transform(X_d_test)
    
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_d_train_s, y_d_train)
    acc = knn.score(X_d_test_s, y_d_test)
    accuracies.append(acc)
    print(f"Dimensions: {dim}, Accuracy: {acc:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(dimensions, accuracies, 'bo-')
plt.xlabel('Number of Dimensions')
plt.ylabel('Accuracy')
plt.title('Curse of Dimensionality: KNN Performance vs Feature Count')
plt.grid(True)
plt.show()